# SentenceKDTransformer on Real mtsamples Transcriptions

This notebook trains the `SentenceKDTransformer` model on the public **mtsamples** corpus (~5,000 real medical transcriptions across ~40 specialties, CC0 public domain) and reproduces four ablation studies that characterise the model's behaviour.

**Paper:** Kim et al. "Integrating ChatGPT into Secure Hospital Networks: A Case Study on Improving Radiology Report Analysis" (CHIL 2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raymondcoding15/PyHealth/blob/DL4H_final/examples/medical_transcriptions_classification_sentence_kd_transformer.ipynb)

**Requires:** CUDA GPU (about 3 to 4 minutes on a Colab T4 with the default 1500-row subsample; set `MAX_SAMPLES=None` in the data cell to run on the full corpus instead, which takes about 10 to 12 minutes) and network access for the one-time mtsamples download (~17 MB) from the public Hugging Face mirror `harishnair04/mtsamples`. No credentials or Kaggle API key are needed.

**Note on the teacher step.** Kim et al. use ChatGPT as a cloud-based teacher to generate ternary sentence labels for MIMIC-CXR. This notebook does not replicate that step for three reasons: (1) `mtsamples` ships with ground-truth specialty labels, so no LLM teacher is needed to train the student on this task; (2) an OpenAI-API teacher would require a paid API key, which prevents the notebook from running end to end for anyone who hasn't set up that account; and (3) for credentialed data like MIMIC-CXR, institutional DUAs generally prohibit sending report text to third-party cloud APIs without a BAA. `SentenceKDTransformer` is teacher-agnostic: any label source (human annotation, a rule-based labeler, or an on-prem LLM such as Llama-3-8B) works unchanged. The ablations below exercise the loss function and the backbone choice, which are the model-level components of the paper.

In [1]:
import importlib
import os
import subprocess
import sys

REPO = "raymondcoding15/PyHealth"
BRANCH = "DL4H_final"

try:
    import google.colab  # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

# 1. Install PyHealth on Colab; no-op when already installed locally.
try:
    import pyhealth  # noqa: F401

    _just_installed = False
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"git+https://github.com/{REPO}.git@{BRANCH}",
        ]
    )
    _just_installed = True

# 2. Colab ships with an older pyarrow pinned into the runtime image.
#    Installing PyHealth's deps can leave pyarrow in a half-upgraded state
#    so pyarrow.lib and pyarrow._csv disagree on struct sizes
#    (`IpcReadOptions size changed, may indicate binary incompatibility`).
#    Detect that at every run and rebuild pyarrow cleanly, regardless of
#    whether we installed PyHealth this session.
_pyarrow_ok = False
if ON_COLAB:
    try:
        import pyarrow.csv  # noqa: F401

        _pyarrow_ok = True
    except Exception as exc:  # pragma: no cover - environment diagnostic
        print(f"pyarrow is in a bad state ({exc}); rebuilding...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "pyarrow"]
        )
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "pyarrow"]
        )

if _just_installed or (ON_COLAB and not _pyarrow_ok):
    print(
        "Packages were (re)installed. On Colab you MUST now do "
        "Runtime -> Restart runtime, then Run all again."
    )

# 2. On Colab the notebook is downloaded standalone, so the companion
#    examples/*.py script isn't on disk. Clone a shallow copy of the repo
#    so the next cell can import the ablation helpers. Locally this is a
#    no-op because the script already sits next to the notebook.
_companion = "medical_transcriptions_classification_sentence_kd_transformer.py"
_local_candidates = [
    os.path.join(os.getcwd(), "examples", _companion),
    os.path.join(os.getcwd(), _companion),
]
if not any(os.path.exists(p) for p in _local_candidates):
    repo_dir = "/content/pyhealth_src"
    if not os.path.isdir(repo_dir):
        subprocess.check_call(
            [
                "git",
                "clone",
                "--quiet",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                f"https://github.com/{REPO}.git",
                repo_dir,
            ]
        )
    print(f"Cloned companion script into {repo_dir}/examples")

  distutils: /private/var/folders/9x/vv1dr63521q6kcn7pxl9s6jc0000gn/T/pip-build-env-9okxyitq/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/9x/vv1dr63521q6kcn7pxl9s6jc0000gn/T/pip-build-env-9okxyitq/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = None
  root = None
  prefix = '/private/var/folders/9x/vv1dr63521q6kcn7pxl9s6jc0000gn/T/pip-build-env-9okxyitq/normal'
  distutils: /private/var/folders/9x/vv1dr63521q6kcn7pxl9s6jc0000gn/T/pip-build-env-9okxyitq/overlay/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/9x/vv1dr63521q6kcn7pxl9s6jc0000gn/T/pip-build-env-9okxyitq/overlay/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = None
  root = None
  prefix = '/private/var/folders/9x/vv1dr63521q6kcn7pxl9s6jc0000gn/T/pip-build-env-9okxyitq/overlay'
ERROR: Coul

CalledProcessError: Command '['/Library/Developer/CommandLineTools/usr/bin/python3', '-m', 'pip', 'install', '-q', 'git+https://github.com/raymondcoding15/PyHealth.git@DL4H_final']' returned non-zero exit status 1.

# 1. Environment Setup

Import the experiment helpers shipped alongside this notebook, confirm a CUDA GPU is available, and seed the RNG for reproducibility.

In [ ]:
import importlib.util
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd
import torch

torch.manual_seed(0)

# This notebook uses the paper's primary backbone (StanfordAIMI/RadBERT,
# ~110M params) on real HuggingFace checkpoints. A GPU is required -- on
# Colab switch to a T4 via: Runtime -> Change runtime type -> T4 GPU.
assert torch.cuda.is_available(), (
    "CUDA is not available. This notebook runs StanfordAIMI/RadBERT "
    "(110M params) and requires a GPU. On Colab go to "
    "Runtime -> Change runtime type -> T4 GPU, then Run all again."
)
DEVICE = torch.device("cuda")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# Locate the companion .py in whichever runtime we're in:
#   - local dev: examples/<script>.py next to this notebook
#   - Colab:     /content/pyhealth_src/examples/<script>.py (cloned above)
here = Path.cwd()
_companion_name = "medical_transcriptions_classification_sentence_kd_transformer.py"
candidates = [
    here / "examples" / _companion_name,
    here / _companion_name,
    Path("/content/pyhealth_src/examples") / _companion_name,
]
script_path = next((p for p in candidates if p.exists()), None)
assert script_path is not None, (
    f"Could not find {_companion_name}. Searched: {[str(p) for p in candidates]}"
)
spec = importlib.util.spec_from_file_location("mt_skd", script_path)
mt_skd = importlib.util.module_from_spec(spec)
# sys.modules registration is required so @dataclass definitions inside the
# loaded module can resolve their own __module__ during class creation.
sys.modules["mt_skd"] = mt_skd
spec.loader.exec_module(mt_skd)
print("Loaded helpers from", script_path)

# 2. Download mtsamples and Instantiate Model

Download the public `harishnair04/mtsamples` corpus from the Hugging Face Hub, stage it as `mtsamples.csv` in a local directory, then construct `SentenceKDTransformer` with the paper's default hyperparameters (`StanfordAIMI/RadBERT` backbone, `lam=1.0`, `temperature=0.07`). The download is cached; subsequent runs skip it.

In [ ]:
from pyhealth.models import SentenceKDTransformer

BACKBONE = "StanfordAIMI/RadBERT"  # paper's primary backbone

# Real mtsamples corpus: ~5,000 transcriptions across ~40 specialties,
# CC0 public domain. We pull it from a Hugging Face mirror rather than
# Kaggle so no API key is required. The download is cached on disk and
# skipped on subsequent notebook runs.
MTSAMPLES_ROOT = Path("/content/mtsamples") if Path("/content").is_dir() else Path.cwd() / "mtsamples_cache"
MTSAMPLES_ROOT.mkdir(parents=True, exist_ok=True)
MTSAMPLES_CSV = MTSAMPLES_ROOT / "mtsamples.csv"

if not MTSAMPLES_CSV.exists():
    from datasets import load_dataset
    print(f"Downloading mtsamples to {MTSAMPLES_CSV} ...")
    hf_ds = load_dataset("harishnair04/mtsamples", split="train")
    hf_ds.to_pandas().to_csv(MTSAMPLES_CSV, index=False)
print(f"mtsamples CSV at: {MTSAMPLES_CSV} "
      f"({MTSAMPLES_CSV.stat().st_size / 1024:.0f} KB)")

# Subsample to 1500 rows so the full ablation suite finishes in a few
# minutes on a Colab T4. Trends are preserved at this size because every
# ablation run sees the same seeded subset. Set MAX_SAMPLES to None to use
# the full ~5000-row corpus.
MAX_SAMPLES = 1500
dataset = mt_skd.build_dataset(
    quick=False,
    data_root=str(MTSAMPLES_ROOT),
    max_samples=MAX_SAMPLES,
)
print(f"Dataset: {len(dataset)} samples, "
      f"{dataset.output_processors['medical_specialty'].size()} classes")

model = SentenceKDTransformer(
    dataset=dataset,
    model_name=BACKBONE,
    lam=1.0,
    temperature=0.07,
    max_length=128,
).to(DEVICE)
print(f"Output classes: {model.get_output_size()} | "
      f"backbone hidden dim: {model.model.config.hidden_size}")

# 3. Ablation 1: Sweep the Contrastive Loss Weight λ

Sweep `lam` across `{0, 0.1, 0.5, 1, 2, 5}` with `temperature=0.07` held constant. Kim et al. (Table 4) only compare `lam=0` against `lam=1`; this sweep covers the full range.

In [ ]:
import contextlib
import io
import logging
import os

# Silence the Trainer's per-step logs, tqdm progress bars, and HuggingFace
# download chatter so the notebook renders cleanly on GitHub. The ablation
# results in the table below are the signal; training chatter from multiple
# fit-evaluate cycles is pure noise here.
logging.disable(logging.CRITICAL)
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

with contextlib.redirect_stdout(io.StringIO()), \
     contextlib.redirect_stderr(io.StringIO()):
    lambda_results = mt_skd.run_lambda_sweep(
        dataset, backbone=BACKBONE, epochs=2, batch_size=16
    )

lambda_df = pd.DataFrame([asdict(r) for r in lambda_results])[
    ["name", "accuracy", "macro_f1", "loss"]
]
lambda_df

# 4. Ablation 2: Sweep the Contrastive Temperature τ

Sweep `temperature` across `{0.05, 0.1, 0.2, 0.5, 1.0}` with `lam=1.0` held constant. This is a novel ablation; the paper uses Khosla et al.'s default of `0.07` without evaluating alternatives.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), \
     contextlib.redirect_stderr(io.StringIO()):
    tau_results = mt_skd.run_temperature_sweep(
        dataset, backbone=BACKBONE, epochs=2, batch_size=16
    )

tau_df = pd.DataFrame([asdict(r) for r in tau_results])[
    ["name", "accuracy", "macro_f1", "loss"]
]
tau_df

# 5. Ablation 3: Learning Rate, Dropout, and Batch Size

Sweep the standard BERT fine-tuning hyperparameters (`lr ∈ {1e-5, 3e-5, 1e-4}`, `dropout ∈ {0.1, 0.3}`, `batch_size ∈ {16, 32}`) with `lam=1.0` and `temperature=0.07` held constant. Results are sorted by accuracy so the winning configuration appears on top.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), \
     contextlib.redirect_stderr(io.StringIO()):
    hparam_results = mt_skd.run_hyperparameter_ablation(
        dataset, backbone=BACKBONE, epochs=2
    )

hparam_df = pd.DataFrame([asdict(r) for r in hparam_results])[
    ["name", "accuracy", "macro_f1", "loss"]
].sort_values("accuracy", ascending=False).reset_index(drop=True)
hparam_df

# 6. Ablation 4: Compare Encoder Backbones under the Contrastive Loss

Train three encoders (`bert-base-uncased`, `emilyalsentzer/Bio_ClinicalBERT`, `StanfordAIMI/RadBERT`) with the full contrastive setup (`lam=1.0`, `temperature=0.07`). Kim et al. (Table 2) compare backbones in the plain cross-entropy setup only.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), \
     contextlib.redirect_stderr(io.StringIO()):
    backbone_results = mt_skd.run_backbone_comparison(
        dataset, quick=False, epochs=2, batch_size=16
    )

backbone_df = pd.DataFrame([asdict(r) for r in backbone_results])[
    ["name", "accuracy", "macro_f1", "loss"]
]
backbone_df

# 7. Save Ablation Results

Persist all four ablation tables to `ablations.json` for downstream reporting and reproducibility.

In [ ]:
import json

all_results = {
    "lambda": [asdict(r) for r in lambda_results],
    "temperature": [asdict(r) for r in tau_results],
    "hyperparam": [asdict(r) for r in hparam_results],
    "backbone": [asdict(r) for r in backbone_results],
}

out_path = Path("ablations.json")
out_path.write_text(json.dumps(all_results, indent=2))
cell_count = sum(len(v) for v in all_results.values())
print(f"Wrote {cell_count} ablation cells to {out_path.resolve()}")
pd.Series({k: len(v) for k, v in all_results.items()}, name="cells")

# 8. Document-Level Aggregation

Apply the paper's per-report aggregation (Eq. 4: max over sentence `p_abnormal`) to a simulated six-sentence chest x-ray report, along with two additional aggregation modes exposed via `doc_agg`: `topk_mean` and `attn`.

In [ ]:
# Demo a 3-way sentence-labeled setup matching the paper's {normal, abnormal, uncertain}.
from pyhealth.datasets import create_sample_dataset

rad_samples = [
    {"patient_id": "p0", "sentence": "no acute intrathoracic abnormality",
     "label": "normal"},
    {"patient_id": "p1", "sentence": "large pleural effusion on the right",
     "label": "abnormal"},
    {"patient_id": "p2", "sentence": "possible early consolidation",
     "label": "uncertain"},
    {"patient_id": "p3", "sentence": "lungs are clear",
     "label": "normal"},
    {"patient_id": "p4", "sentence": "opacity consistent with pneumonia",
     "label": "abnormal"},
    {"patient_id": "p5", "sentence": "unclear if atelectasis vs effusion",
     "label": "uncertain"},
]
rad_dataset = create_sample_dataset(
    samples=rad_samples,
    input_schema={"sentence": "text"},
    output_schema={"label": "multiclass"},
    dataset_name="radiology-sentences-demo",
)
rad_model = SentenceKDTransformer(
    dataset=rad_dataset,
    model_name=BACKBONE,
    lam=1.0,
    max_length=64,
).to(DEVICE)

report = [s["sentence"] for s in rad_samples]
print("Report sentences:")
for i, s in enumerate(report):
    print(f"  [{i}] {s}")

rows = []
for mode in ("max", "topk_mean", "attn"):
    doc = rad_model.document_predict(report, doc_agg=mode)
    rows.append({
        "aggregation": mode,
        "p_abnormal (document)": round(doc["pa"], 4),
        "p_normal (document)": round(doc["pn"], 4),
        "abnormal_class_index": doc["abnormal_index"],
    })

doc_df = pd.DataFrame(rows)
doc_df

In [ ]:
# Per-sentence ternary probabilities that feed the aggregations above.
doc = rad_model.document_predict(report, doc_agg="max")
per_sent = doc["per_sentence_probs"].cpu().numpy()

vocab = rad_dataset.output_processors["label"].label_vocab
columns = [None] * len(vocab)
for label, idx in vocab.items():
    columns[idx] = f"p({label})"

sent_df = pd.DataFrame(per_sent, columns=columns)
sent_df.insert(0, "sentence", report)
sent_df.round(4)

# 9. Next Steps

Run the headless version of these ablations against the same real mtsamples corpus this notebook downloaded:

```bash
python examples/medical_transcriptions_classification_sentence_kd_transformer.py \
    --data_root /content/mtsamples --epochs 3
```

The companion script also supports a fast synthetic mode for quick smoke tests (no download required):

```bash
python examples/medical_transcriptions_classification_sentence_kd_transformer.py --quick --epochs 1
```

Use the model in a PyHealth pipeline:

```python
from pyhealth.models import SentenceKDTransformer
model = SentenceKDTransformer(
    dataset=sample_dataset,
    model_name="StanfordAIMI/RadBERT",
    lam=1.0,
    temperature=0.07,
)
```

Aggregate sentence-level predictions into report-level scores:

```python
pa, pn, probs = model.document_predict(list_of_sentences, doc_agg="max")
```